# M3 step 9: inpainting (task 5, RQ2 AI edit)

Inpainting uses AI to redraw part of a photo, for example to remove an object. Only part of the stamp
is damaged, so the expected result lies between the normal edits and the full VAE round trip.

For each stamped photo and each mask size (about 10%, 25% and 50% of the photo):

1. a rectangle of that area is placed at a fixed random position (seeded per photo and size),
2. `stable-diffusion-v1-5/stable-diffusion-inpainting` redraws it with the neutral prompt "a natural photo",
   30 steps, guidance 7.5 and a fixed seed, at 512 by 512,
3. the result is resized back to the original size.

**Two versions of each result are read.** The inpainting model returns a whole new image: even the
pixels outside the mask have passed through its VAE, which on its own breaks the stamp (notebook 8).
Photo editors paste only the redrawn region back into the photo. So:

- **composited** (the main result, the guide's test): original pixels outside the mask, new pixels inside;
- **raw model output**: what a simple tool that returns the model's image as it is would produce.

Expected: bit accuracy drops as the masked area grows (composited); the raw output should behave like
the VAE round trip at every mask size.

**GPU:** required. About 3 to 4 seconds per inpainting on a T4, so 200 photos x 3 sizes is roughly
40 to 60 minutes, and all 500 photos about 2 hours. Results are saved after every photo, so a
disconnected session continues where it stopped. The guide allows a fixed subset of at least 200
photos; say which one you used in the report.

## Setup

**Kaggle:** turn on *Settings > Accelerator > GPU T4 x2* (only notebooks 8, 9 and 11 need it) and
*Internet on*, and attach the `veristamp-m3-inputs` dataset (README.md, "Running on Kaggle"). The first
cell then finds the code, installs what is missing and moves into `m3_experiments`. No path to edit.

**Colab or a local machine:** set the paths once in `settings.py`, then run from the `m3_experiments`
folder.

In [ ]:
# Kaggle setup: find the dataset wherever Kaggle put it, copy the code, point the notebooks at the files.
import os, shutil, subprocess, sys
from pathlib import Path

def find(rel, max_depth=6):
    for depth in range(max_depth + 1):
        hits = sorted(Path('/kaggle/input').glob('/'.join(['*'] * depth + [rel])))
        if hits:
            return hits[0]
    return None

if Path('/kaggle/input').is_dir() and not Path('settings.py').exists():
    code = find('m3_experiments/settings.py')
    model = find('m3/best.pt')
    print('Code found at: ', code)
    print('Model found at:', model)
    if code is None or model is None:
        print(subprocess.run(['find', '/kaggle/input', '-maxdepth', '4'], capture_output=True, text=True).stdout[:3000])
        raise SystemExit('The veristamp-m3-inputs dataset is not attached, or has a different layout (see the list above).')
    repo = Path('/kaggle/working/VeriStamp')
    if not (repo / 'm3_experiments' / 'settings.py').exists():
        shutil.rmtree(repo, ignore_errors=True)          # removes a failed earlier clone
        shutil.copytree(code.parent.parent, repo, ignore=shutil.ignore_patterns('.git'))
    # Each item is found on its own, so the inputs can come in several zips or datasets.
    paths = {'M3E_MODEL_PATH': 'm3/best.pt', 'M3E_TEST_DIR': 'm3/test', 'M3E_TEST_MANIFEST': 'm3/test_manifest.csv',
             'M3E_GENIMAGE_SPLIT_DIR': 'genimage/splits', 'M3E_GENIMAGE_ROOT': 'genimage/images',
             'M3E_M1_CHECKPOINT': 'm1/best.pt', 'M3E_M1_VSLIB_DIR': 'm1/vslib_m1_final'}
    for name, rel in paths.items():
        hit = find(rel)
        os.environ[name] = str(hit if hit else model.parent.parent / rel)   # the notebooks run from here inherit these
        print(f'  {rel:<22} {"found" if hit else "not in the inputs"}')
    os.environ['M3E_WORK_DIR'] = '/kaggle/working/m3_work'
    os.chdir(repo / 'm3_experiments')
    for pkg in ('diffusers', 'lpips'):
        try:
            __import__(pkg)
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
print('Working in', Path.cwd())

In [ ]:
# On Colab instead, uncomment and run once per session:
# from google.colab import drive; drive.mount('/content/drive')
# !git clone https://github.com/KruthikaVirupakshappa/VeriStamp.git /content/VeriStamp
# %cd /content/VeriStamp/m3_experiments
# !pip install -q diffusers transformers accelerate scikit-image lpips

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt

import settings as S
import m3_eval as E
mw = S.setup()                      # imports ../m3_watermark/m3_watermark.py as it is
torch.manual_seed(S.SEED)
np.random.seed(S.SEED)

m3 = mw.load_m3(S.MODEL_PATH)       # load once, reuse for every photo
DEVICE = m3.device
print('Model epoch', m3.epoch, 'on', DEVICE)

In [ ]:
files = E.list_photos(S.TEST_DIR, S.N_TEST_PHOTOS)
if len(files) != S.N_TEST_PHOTOS:
    raise RuntimeError(f'Expected {S.N_TEST_PHOTOS} test photos in {S.TEST_DIR}, found {len(files)}')
# Every photo gets a fixed verdict, P(AI) and tag (E.payload_for), the same in every notebook.
# The stamped PNGs are made once and reused by notebooks 6 to 9.
records = E.stamp_test_set(mw, m3, files, S.WORK_DIR, S.MODEL_PATH, S.SEED)
print(len(records), 'stamped photos in', S.WORK_DIR / 'stamped')
records.head(3)

## Settings and the fixed subset

In [ ]:
N_INPAINT = 200            # at least 200; set to 500 if there is time. The first 200 stay the same.
FRACTIONS = [0.10, 0.25, 0.50]
STEPS, GUIDANCE = 30, 7.5

order = records.sample(frac=1, random_state=S.SEED).reset_index(drop=True)   # fixed order
subset = order.head(N_INPAINT)
progress_file = S.WORK_DIR / 'inpainting_progress.csv'
done = pd.read_csv(progress_file) if progress_file.exists() else pd.DataFrame(columns=['name', 'fraction'])
done_keys = set(zip(done['name'], done['fraction'].astype(float).round(2)))
print(f'{len(subset)} photos, {len(done_keys)} of {len(subset) * len(FRACTIONS)} inpaintings already done')

In [ ]:
pipe = E.load_inpainter(DEVICE, S.INPAINT_NAME)
print('Inpainting model loaded:', S.INPAINT_NAME)

## Run (safe to stop and restart)

In [ ]:
import time
example_dir = S.WORK_DIR / 'inpainting_examples'
example_dir.mkdir(exist_ok=True)
start = time.time()
for i, (_, r) in enumerate(E.tqdm(list(subset.iterrows()))):
    for frac in FRACTIONS:
        if (r['name'], round(frac, 2)) in done_keys:
            continue
        stamped = E.open_rgb(r['stamped'])
        w, h = stamped.size
        box = E.mask_box(w, h, frac, r['name'], S.SEED)
        seed = E.photo_seed(r['name'], f'inpaint{frac}', S.SEED)
        composited, raw = E.inpaint(pipe, stamped, box, S.INPAINT_PROMPT, seed, STEPS, GUIDANCE)
        if i < 3:
            composited.save(example_dir / f"{Path(r['name']).stem}_{int(frac * 100)}_composited.png")
            raw.save(example_dir / f"{Path(r['name']).stem}_{int(frac * 100)}_raw.png")
        actual = (box[2] - box[0]) * (box[3] - box[1]) / (w * h)
        new = []
        for variant, im in [('composited', composited), ('raw model output', raw)]:
            res = E.score_read(mw, m3, im, r['expected_bits'], r['verdict'])
            new.append({'name': r['name'], 'fraction': frac, 'actual_fraction': actual, 'variant': variant,
                        'box': str(box), **res})
        pd.DataFrame(new).to_csv(progress_file, mode='a', header=not progress_file.exists(), index=False)
        done_keys.add((r['name'], round(frac, 2)))
print(f'{time.time() - start:.0f} s this session')

## Results

In [ ]:
per_photo = pd.read_csv(progress_file)
per_photo = per_photo[per_photo['name'].isin(subset['name'])]
per_photo['edit'] = per_photo['fraction'].map(lambda f: f'Inpainting, {int(round(f * 100))}% mask')
n_complete = per_photo.groupby('name')['fraction'].nunique().eq(len(FRACTIONS)).sum()
print('Photos with all mask sizes done:', n_complete, 'of', len(subset))
summary = E.summarise(per_photo, by=('edit', 'variant'))
print('Mean mask area actually covered:', per_photo.groupby('edit')['actual_fraction'].mean().round(3).to_dict())
E.as_percent(summary)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for variant, g in summary.groupby('variant'):
    ax.plot([10, 25, 50], 100 * g['bit_accuracy'].values, marker='o', label=f'{variant}: bits')
    ax.plot([10, 25, 50], 100 * g['valid_verdict'].values, marker='s', linestyle='--', label=f'{variant}: valid verdict')
ax.axhline(50, color='grey', linewidth=0.8)
ax.set_xlabel('Mask area (% of photo)'); ax.set_ylabel('%'); ax.set_ylim(0, 102); ax.legend(fontsize=8)
ax.set_title('Stamp after inpainting')
plt.tight_layout()
(S.RESULTS_DIR / 'figures').mkdir(exist_ok=True)
plt.savefig(S.RESULTS_DIR / 'figures' / 'rq2_inpainting.png', dpi=120, bbox_inches='tight')
plt.show()

## Save

In [ ]:
E.save_results(S.RESULTS_DIR, 'rq2_inpainting', per_photo=per_photo, summary=summary,
               meta={'model': S.INPAINT_NAME, 'prompt': S.INPAINT_PROMPT, 'steps': STEPS, 'guidance': GUIDANCE,
                     'n_photos': int(n_complete), 'subset': f'first {N_INPAINT} of the 500 test photos after a '
                     f'shuffle with seed {S.SEED}', 'fractions': FRACTIONS, 'size': 512,
                     'environment': E.environment_info()})
print(E.to_markdown(E.as_percent(summary)))